# 00 — Setup & Data Download
**Project:** ViT Reliability & Explainability Under Medical Distribution Shift
**Author:** Sosna Worku

---
### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Have your Kaggle key ready (kaggle.com → Settings → API → Create New Token)

### Run this notebook only ONCE — data stays in Google Drive permanently.

## Step 1 — Clone GitHub repo

In [ ]:
import os, sys

REPO      = 'vit-medical-shift'
REPO_PATH = f'/content/{REPO}'
GITHUB    = 'https://github.com/sossyh/vit-medical-shift.git'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull origin main')
else:
    print('Cloning repo...')
    os.system(f'git clone {GITHUB} {REPO_PATH}')

sys.path.insert(0, REPO_PATH)
print('Done! Repo ready at', REPO_PATH)

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## Step 3 — Verify GPU

In [ ]:
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name      :', torch.cuda.get_device_name(0))
    print('GPU memory    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Step 4 — Install packages

In [ ]:
!pip install -q timm torchmetrics grad-cam einops pyyaml kaggle
print('All packages installed!')

## Step 5 — Set up Kaggle credentials
1. Go to kaggle.com → profile picture → Settings → API → Create New Token
2. Copy only the key string (the part after KGAT_)
3. Paste it below

In [ ]:
import os, json

# ── Fill in your key below ─────────────────────────────────
KAGGLE_USERNAME = 'sosnaworku'
KAGGLE_KEY      = 'paste_your_key_here'   # only the part after KGAT_
# ───────────────────────────────────────────────────────────

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials saved!')

# Test — should show list of files in the dataset
!kaggle datasets files nih-chest-xrays/data

## Step 6 — Download NIH ChestX-ray14
Downloads ~7GB (2 image zips = ~18k images).  
Takes 20-30 minutes — **do not close Colab.**

In [ ]:
from importlib import reload
import src.download_data
reload(src.download_data)
from src.download_data import download_nih

download_nih(
    drive_root='/content/drive/MyDrive/data/nih',
    n_image_zips=2
)

## Step 6b — If images still show 0, run this to fix extraction

In [ ]:
# Run this ONLY if Step 6 shows 0 images
import os, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/data/nih'
IMG_DIR    = f'{DRIVE_ROOT}/images'
os.makedirs(IMG_DIR, exist_ok=True)

for i in range(1, 3):  # zips 001 and 002
    zip_name = f'images_{i:03d}.tar.gz'
    zip_path = f'{DRIVE_ROOT}/{zip_name}'

    # Re-download if missing
    if not os.path.exists(zip_path):
        print(f'Re-downloading {zip_name}...')
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'nih-chest-xrays/data',
            '-f', zip_name,
            '--path', DRIVE_ROOT
        ])

    # Extract with verbose output
    print(f'Extracting {zip_name}...')
    result = subprocess.run(
        ['tar', '-xvzf', zip_path, '-C', IMG_DIR],
        capture_output=True, text=True
    )
    # Show first few extracted files
    lines = result.stdout.strip().split('\n')
    print(f'  First extracted files: {lines[:3]}')
    if result.returncode != 0:
        print(f'  Error: {result.stderr[:200]}')

    # Remove zip
    if os.path.exists(zip_path):
        os.remove(zip_path)

# Final count
all_files = os.listdir(IMG_DIR)
print(f'\nTotal images: {len(all_files):,}')
print(f'Sample files: {all_files[:5]}')

## Step 7 — Verify everything

In [ ]:
from src.download_data import verify_data
verify_data('/content/drive/MyDrive/data/nih')

## Done!
Data is permanently saved in Google Drive — never run this notebook again.

**Next step:** Open `01_data_exploration.ipynb`